In [17]:
import kagglehub
path = kagglehub.dataset_download("andrewmvd/cyberbullying-classification")

Using Colab cache for faster access to the 'cyberbullying-classification' dataset.


In [18]:
!pip install langdetect
!pip install emoji
!pip install nltk

In [19]:
import pandas as pd
path = path + "/cyberbullying_tweets.csv"

In [20]:
# import Check dataset
df = pd.read_csv(path)
print(df.shape)
print(df.columns)
df.head()

(47692, 2)
Index(['tweet_text', 'cyberbullying_type'], dtype='object')


,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying


In [21]:
# Find unique values in the 'category' column
categories = df['cyberbullying_type'].unique()
print(categories)

['not_cyberbullying' 'gender' 'religion' 'other_cyberbullying' 'age'
 'ethnicity']


Pre- processing

In [22]:
#lowercase conversion
df['tweet_text'] = df['tweet_text'].str.lower()


In [23]:
import emoji

# Find tweets containing at least one emoji
emoji_tweets = df[df['tweet_text'].apply(emoji.emoji_count) > 0]

emoji_tweets.head()

,tweet_text,cyberbullying_type
21,kids love😘❤ @ mohamad bin zayed city مدينة محم...,not_cyberbullying
80,@imrbarfield it starts thursday. my classes ar...,not_cyberbullying
135,mmmm #mkr forget deconstruction @lisamromano @...,not_cyberbullying
144,rt @mykitchenrules: nawwww 😭😭😭 #mkr,not_cyberbullying
151,@justinbieber your how i get through bullying ...,not_cyberbullying


In [24]:
#emoji to text conversion
df['tweet_text'] = df['tweet_text'].apply(emoji.demojize)
print(df.loc[21])
df.head()

tweet_text            kids love:face_blowing_a_kiss::red_heart: @ mo...
cyberbullying_type                                    not_cyberbullying
Name: 21, dtype: object


,tweet_text,cyberbullying_type
0,"in other words #katandandre, your food was cra...",not_cyberbullying
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying


In [25]:
# Remove missing/null tweets
df = df.dropna(subset=['tweet_text'])

# Convert to string
df['tweet_text'] = df['tweet_text'].astype(str)

# Remove empty or whitespace-only tweets
df = df[df['tweet_text'].str.strip() != '']

# Remove duplicate tweets
df = df.drop_duplicates(subset=['tweet_text'])

# Reset index
df = df.reset_index(drop=True)

print("Remaining tweets:", len(df))

Remaining tweets: 45991


In [26]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\w+', '', text)

    # Remove special characters and numbers and all non-ASCII characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()


    return text

# Apply cleaning
df['clean_tweet'] = df['tweet_text'].apply(clean_text)
df.head()

,tweet_text,cyberbullying_type,clean_tweet
0,"in other words #katandandre, your food was cra...",not_cyberbullying,in other words katandandre your food was crapi...
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,why is aussietv so white mkr theblock imaceleb...
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,a classy whore or more red velvet cupcakes
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks for the heads up but not too conc...
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,this is an isis account pretending to be a kur...


In [27]:
# Remove tweets that are extremely short
df = df[df['clean_tweet'].str.len() >= 3]
print("Remaining tweets:", len(df))

Remaining tweets: 45701


In [28]:
# Detect the language
from langdetect import detect, LangDetectException

def detect_language(text):
    try:
        return detect(str(text))
    except LangDetectException:
        return 'unknown'

df['language'] = df['clean_tweet'].apply(detect_language)
# See which tweets are NOT English
non_english = df[df['language'] != 'en']
print(non_english[['clean_tweet', 'language']])

                                             clean_tweet language
6      itu sekolah ya bukan tempat bully ga jauh kaya...       id
12                                            ughhhh mkr       sw
15     parem de fazer bullying comigo uhahuah bando d...       pt
16              tadinhu de mim sofrendo bulling viu mimi       pt
39                                              fuck mkr       da
...                                                  ...      ...
45935  you fucken dumb muthafucka nigger i hope you d...       da
45946                       lmao nigga dont you got work       af
45950  im dark skin you dumb fuck rt you need to die ...       af
45953                 fuck obama dumb ass nigger mad ehh       et
45973                      fuck you doris ya dumb nigger       id

[3031 rows x 2 columns]


In [29]:
print(non_english['cyberbullying_type'].value_counts())

cyberbullying_type
not_cyberbullying      1391
other_cyberbullying     705
ethnicity               508
gender                  384
religion                 26
age                      17
Name: count, dtype: int64


In [30]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

# English stopwords
stop_words = set(stopwords.words('english'))

def clean_text(text):

    # Remove stopwords
    text = ' '.join(
        word for word in text.split()
        if word not in stop_words
    )

    return text

# Apply cleaning
df['clean_tweet'] = df['clean_tweet'].apply(clean_text)
df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,tweet_text,cyberbullying_type,clean_tweet,language
0,"in other words #katandandre, your food was cra...",not_cyberbullying,words katandandre food crapilicious mkr,en
1,why is #aussietv so white? #mkr #theblock #ima...,not_cyberbullying,aussietv white mkr theblock imacelebrityau tod...,en
2,@xochitlsuckkks a classy whore? or more red ve...,not_cyberbullying,classy whore red velvet cupcakes,en
3,"@jason_gio meh. :p thanks for the heads up, b...",not_cyberbullying,meh p thanks heads concerned another angry dud...,en
4,@rudhoeenglish this is an isis account pretend...,not_cyberbullying,isis account pretending kurdish account like i...,en


In [33]:
# Apply lemmatization
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return ' '.join(words)

df['lemmatized_tweet'] = df['clean_tweet'].apply(lemmatize_text)

df[['clean_tweet', 'lemmatized_tweet']].head()

LookupError: 
**********************************************************************
  Resource [93mwordnet[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('wordnet')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mcorpora/wordnet[0m

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


VADER was selected because it is lightweight, interpretable, designed for short informal social-media text, and provides continuous sentiment features that can be incorporated alongside TF-IDF features without requiring computationally expensive language models.

In [ ]:
import nltk
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

analyzer = SentimentIntensityAnalyzer()

# Calculate VADER sentiment scores
df['sentiment_scores'] = df['lemmatized_tweet'].apply(
    analyzer.polarity_scores
)

# Extract four sentiment scores
df['negative'] = df['sentiment_scores'].apply(lambda x: x['neg'])
df['positive'] = df['sentiment_scores'].apply(lambda x: x['pos'])
df['neutral']  = df['sentiment_scores'].apply(lambda x: x['neu'])
df['compound'] = df['sentiment_scores'].apply(lambda x: x['compound'])

# Keep only negative-polarity tweets
df = df[df['compound'] < 0].copy()

# Reset index
df.reset_index(drop=True, inplace=True)


print("Remaining tweets:", len(df))
df[['lemmatized_tweet','negative','positive','neutral','compound']].head()